# Formix verified FormL chatbot

This notebook runs the passing Formix adapter entirely on a Kaggle GPU. Each prompt generates a complete FormL program, checks it with the native C++ compiler, and attempts up to two targeted repairs. Type EXIT to end the chatbot.

Enable a Kaggle GPU and Internet. Attach the passing Formix artifacts as a Kaggle Model or attach formix-v2-passed-artifacts.zip as input.

In [ ]:
from pathlib import Path
import os, subprocess

REPO = Path("/kaggle/working/formix")
BRANCH = "codex/formix-model-v2"
if not (REPO / ".git").is_dir():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        "https://github.com/imrishi007/formix.git", str(REPO),
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
os.chdir(REPO)
print("Repository ready:", REPO)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "ai-training/requirements.txt",
], check=True)

import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU, restart the session, and rerun."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import json, zipfile

ADAPTER = REPO / "ai-training/output-v2/adapter"
gate_path = ADAPTER.parent / "quality_gate.json"

if not (ADAPTER / "adapter_config.json").is_file():
    mounted = []
    for root in (Path("/kaggle/input"), Path("/kaggle/models")):
        if root.exists():
            mounted.extend(config.parent for config in root.rglob("adapter_config.json"))
    mounted = [
        path for path in mounted
        if (path / "adapter_model.safetensors").is_file()
        and (path.parent / "quality_gate.json").is_file()
    ]
    passing = []
    for path in mounted:
        gate = json.loads((path.parent / "quality_gate.json").read_text())
        if gate.get("passed") is True:
            passing.append(path)
    if passing:
        ADAPTER = passing[0]
        gate_path = ADAPTER.parent / "quality_gate.json"
        print("Using mounted Kaggle Model:", ADAPTER)

if not (ADAPTER / "adapter_config.json").is_file():
    archives = list(Path("/kaggle/input").rglob("*artifacts*.zip"))
    archives += list(Path("/kaggle/working").glob("*artifacts*.zip"))
    selected = None
    for archive_path in archives:
        try:
            with zipfile.ZipFile(archive_path) as archive:
                gate_names = [
                    name for name in archive.namelist()
                    if name.replace("\\", "/").endswith("ai-training/output-v2/quality_gate.json")
                ]
                if gate_names and json.loads(archive.read(gate_names[0])).get("passed") is True:
                    selected = archive_path
                    break
        except (zipfile.BadZipFile, json.JSONDecodeError):
            pass
    if selected is None:
        raise FileNotFoundError("No passing Formix adapter was found in Kaggle inputs.")
    with zipfile.ZipFile(selected) as archive:
        repo_root = REPO.resolve()
        for member in archive.infolist():
            destination = (REPO / member.filename).resolve()
            if destination != repo_root and repo_root not in destination.parents:
                raise RuntimeError("Unsafe archive member: " + member.filename)
        archive.extractall(REPO)
    ADAPTER = REPO / "ai-training/output-v2/adapter"
    gate_path = ADAPTER.parent / "quality_gate.json"
    print("Extracted:", selected)

config = json.loads((ADAPTER / "adapter_config.json").read_text())
gate = json.loads(gate_path.read_text())
weights = ADAPTER / "adapter_model.safetensors"
assert gate.get("passed") is True
assert weights.stat().st_size > 200_000_000
print("Adapter verified:", ADAPTER)
print("Base model:", config["base_model_name_or_path"])

In [ ]:
COMPILER = REPO / "forml-compiler/build/forml_compiler"
COMPILER.parent.mkdir(parents=True, exist_ok=True)

compiler_sources = [
    "forml-compiler/src/token.cpp",
    "forml-compiler/src/diagnostics.cpp",
    "forml-compiler/src/lexer.cpp",
    "forml-compiler/src/ast.cpp",
    "forml-compiler/src/parser.cpp",
    "forml-compiler/src/semantic_analyzer.cpp",
    "forml-compiler/src/json_serializer.cpp",
    "forml-compiler/src/forml_bridge.cpp",
    "forml-compiler/examples/main.cpp",
]

subprocess.run([
    "g++", "-std=c++17", "-O2",
    "-I", "forml-compiler/include",
    "-I", "forml-compiler/third_party",
    *compiler_sources,
    "-o", str(COMPILER),
], cwd=REPO, check=True)

print("Native compiler ready:", COMPILER)

In [ ]:
sys.path.insert(0, str(REPO / "ai-training"))

from demo_local import (
    compile_source,
    generate,
    load_model,
    repair_prompt,
)

model, tokenizer = load_model(ADAPTER)
print("\nFormix chatbot is ready.")
print("Type a form request and press Enter.")
print("Type EXIT to stop.")

In [ ]:
LATEST_SOURCE = Path("/kaggle/working/latest_generated.forml")

while True:
    try:
        request = input("\nForm request (EXIT to stop)> ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nChat stopped.")
        break

    if request.upper() == "EXIT":
        print("Chat stopped.")
        break

    if not request:
        continue

    print("\nGenerating and checking FormL...")
    source = generate(
        model,
        tokenizer,
        request,
        max_new_tokens=700,
    )
    result = compile_source(COMPILER, source)
    repairs_used = 0

    for attempt in range(1, 3):
        if result.get("success"):
            break

        repairs_used = attempt
        print("Compiler rejected the candidate; repair", attempt, "of 2")

        source = generate(
            model,
            tokenizer,
            repair_prompt(
                source,
                result.get("diagnostics", []),
                attempt,
            ),
            max_new_tokens=700,
        )
        result = compile_source(COMPILER, source)

    LATEST_SOURCE.write_text(source, encoding="utf-8")

    print("\n" + "=" * 80)
    print("GENERATED FORML")
    print("=" * 80)
    print(source)
    print("=" * 80)

    if result.get("success"):
        print("COMPILER VERIFIED: PASS")
        print("Repairs used:", repairs_used)
    else:
        print("COMPILER VERIFIED: FAIL")
        for item in result.get("diagnostics", []):
            print(
                str(item.get("severity", "error")).upper(),
                str(item.get("line", 0)) + ":" + str(item.get("column", 0)),
                item.get("message", ""),
            )

    print("Latest source saved to:", LATEST_SOURCE)